# Part 5 · 00 Setup and input check

This notebook runs once before anything else in part 5. It:

1. records the environment (Python and package versions, for the reproducibility checklist);
2. checks that every input from parts 1–2 and 3–4 is where part 5 expects it, and says which mode `cache_raw.py` will run in;
3. runs the unit tests of the cleaning on four kinds of document (no data, no network).

Part 5 needs neither WRDS nor any login: the WRDS data it uses (CRSP names, I/B/E/S link) were already pulled by part 3–4.

**Before running:** clone the repository and open this notebook inside it. Set `TEAM_DIR` if the private data live outside the checkout.

**Folder layout**

```
S08/5_data cleaning/
  code/     settings.py (paths), cache_raw, clean_release (+ tests), entities,
            token_stats, dedup, model_cache, splits, manifest
  cache/    raw EDGAR bytes (raw/raw_YYYY.zip + raw_manifest.parquet) and model outputs
  data/     built tables keyed by accession: clean/, chunks/ (pieces per model), entities,
            token_stats, dedup_flags, splits
  results/  tables for the report
  logs/     environment records
```

**Inputs read, never changed:** `files csv` (event table, extraction report, timing), `scripts/fetch_releases.py`, and from `3_4_surprise_returns` the release clock, the trading calendar, `crsp_names.parquet`, `link_ibes_crsp.parquet` and `SPLITS` in `common.py`.

In [ ]:
%pip install -q beautifulsoup4 lxml nltk pyarrow datasketch tiktoken transformers


In [ ]:
import importlib
import os
import sys
from pathlib import Path

_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts" / "project_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook inside S08, or set S08_ROOT to the checkout path.")
sys.path.insert(0, str(REPO_ROOT / "scripts"))
import project_paths
importlib.reload(project_paths)
from project_paths import (REPO_ROOT, TEAM_DIR, SCRIPTS, NOV2_DATA,
                           PROJECT_DIR, STEP5_DIR, PART34_CODE, STEP5_CODE)

# Notebook paths.
CODE = STEP5_CODE
STEP5_DIR.mkdir(parents=True, exist_ok=True)

for sub in ["cache", "data", "results", "logs"]:
    (STEP5_DIR / sub).mkdir(parents=True, exist_ok=True)

os.environ["TEAM_DIR"] = str(TEAM_DIR)
os.environ["STEP5_DIR"] = str(STEP5_DIR)
sys.path.insert(0, str(CODE))

missing = [f for f in ["settings.py", "cache_raw.py", "clean_release.py", "test_clean_release.py",
                                 "entities.py", "token_stats.py", "dedup.py", "model_cache.py",
                                 "splits.py", "manifest.py"] if not (CODE / f).exists()]
assert not missing, f"Missing code in {CODE}: {missing}"
print("step 5 folder:", STEP5_DIR)
print("code files:   ", sorted(os.listdir(CODE)))

In [ ]:
# Environment record: goes into logs/ and, later, the README.
import json, platform, datetime as dt
import importlib.metadata as md

env = {"run_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
       "python": platform.python_version()}
for pkg in ["pandas", "numpy", "pyarrow", "beautifulsoup4", "lxml", "nltk",
            "datasketch", "tiktoken", "transformers"]:
    try:
        env[pkg] = md.version(pkg)
    except md.PackageNotFoundError:
        env[pkg] = "not installed"
(STEP5_DIR / "logs" / f"env_{dt.date.today()}.json").write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))

## Inputs from the other parts

Every file part 5 reads, with its size. `extraction_report_full.csv` is the full report of `fetch_releases.py` (with `accession`, `cik` and `filename`); the shared `extraction_report.csv` has only `source`, `words` and `tables_dropped`. With the full report, `cache_raw.py` makes one request per document (3 to 4 hours); without it, it reads each index page again (7 to 8 hours).

In [ ]:
import pandas as pd
import settings as S

inputs = {
    "event table (1_2nov)": S.EVENT_TABLE,
    "extraction report, shared (1_2nov)": S.EXTRACTION_REPORT,
    "three-source timing (1_2nov)": S.TIMING,
    "fetch_releases.py (1_2nov)": S.NOV2_CODE / "fetch_releases.py",
    "release clock (3_4)": S.CLOCK,
    "trading calendar (3_4)": S.CALENDAR,
    "CRSP names (3_4 cache)": S.CRSP_NAMES,
    "I/B/E/S-CRSP link (3_4 cache)": S.IBES_LINK,
    "common.py with SPLITS (3_4)": S.PART34_CODE / "common.py",
}
for label, p in inputs.items():
    print(f"  {'ok ' if p.exists() else 'MISSING'}  {label:<38} "
          f"{p.stat().st_size if p.exists() else 0:>12,}")

full = S.extraction_full()
print()
print("cache_raw.py mode:", f"report ({full})" if full else
      "index (no extraction_report_full.csv: each index page is read again)")

clk = pd.read_csv(S.CLOCK, usecols=["accession", "reaction_date", "status"], dtype=str)
print(f"\nrelease clock: {len(clk):,} rows, {clk.accession.nunique():,} accessions, "
      f"{clk.reaction_date.notna().sum():,} with a reaction date")
print(clk.status.value_counts().to_string())
print("\nsplits from common.py:", S.common_3_4().SPLITS)

## Unit tests of the cleaning

Four small documents built like the four kinds met in the sample (a recent HTML release, an old release laid out in a table, an old `.txt` filing in SGML, an iXBRL 8-K body). Every line should start with `ok`.

In [ ]:
__import__("subprocess").run([sys.executable, str(CODE / 'test_clean_release.py')], check=True)